In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── 데이터 로드 ──────────────────────────────────────────
df = pd.read_csv('hallucination_error_detail.csv', encoding='utf-8-sig')
print(f"전체 오류 레코드: {len(df)}건")
print(f"컬럼: {df.columns.tolist()}")

# ── Extraction Failure만 필터링 ──────────────────────────
ef = df[df['오류원인'] == 'Extraction Failure'].copy()
print(f"\nExtraction Failure 총계: {len(ef)}건")

# ── 유형 분류 ────────────────────────────────────────────
def classify_ef(row):
    val = row['추출값']
    # 1) JSON 파싱 실패: 추출값 자체가 NaN
    if pd.isna(val):
        return 'null/empty 반환'
    val_str = str(val).strip()
    # 2) 빈 문자열
    if val_str == '' or val_str.lower() in ('none', 'null', 'nan'):
        return 'null/empty 반환'
    # 3) 수치형인데 문자열로 반환
    if row['수치형']:
        try:
            float(val_str.replace(',', '.'))
            return '형식 불일치(문자→숫자 파싱 성공)'
        except:
            return '형식 불일치(숫자 대신 문자열)'
    # 4) 범주형 불일치
    return '형식 불일치(범주형 불일치)'

ef['세부유형'] = ef.apply(classify_ef, axis=1)

# ── 수치형/범주형 구분 추가 ──────────────────────────────
ef['변수유형'] = ef['수치형'].map({True: '연속형', False: '범주형'})

# ── 결과 출력 ────────────────────────────────────────────
print("\n" + "=" * 65)
print("Extraction Failure 세부 유형 분류 (N=519건)")
print("=" * 65)

type_dist = ef['세부유형'].value_counts()
for t, cnt in type_dist.items():
    print(f"  {t}: {cnt}건 ({cnt/len(ef)*100:.1f}%)")

print("\n▶ 변수 유형별 분포")
cross = pd.crosstab(ef['세부유형'], ef['변수유형'])
print(cross)

print("\n▶ 상위 발생 변수 (Extraction Failure 기준)")
var_dist = ef['변수명'].value_counts().head(10)
for var, cnt in var_dist.items():
    print(f"  {var}: {cnt}건 ({cnt/len(ef)*100:.1f}%)")

print("\n▶ 대응 방안 제언")
print("  null/empty 반환    → 프롬프트 개선 (값 필수 반환 지시 강화)")
print("  형식 불일치(수치형) → Guided Decoding(--guided-json) 적용 권장")
print("  형식 불일치(범주형) → 허용값 목록(Enum) 명시 또는 후처리 정규화")

# ── 저장 ─────────────────────────────────────────────────
ef.to_csv('extraction_failure_classified.csv', index=False, encoding='utf-8-sig')
print("\n결과 저장: extraction_failure_classified.csv")

전체 오류 레코드: 2009건
컬럼: ['개인아이디', '정보손실률', '변수명', '원본값', '추출값', '오차', '수치형', '환각유형', '오류원인']

Extraction Failure 총계: 519건

Extraction Failure 세부 유형 분류 (N=519건)
  null/empty 반환: 519건 (100.0%)

▶ 변수 유형별 분포
변수유형           범주형
세부유형              
null/empty 반환  519

▶ 상위 발생 변수 (Extraction Failure 기준)
  아침식사 빈도: 130건 (25.0%)
  평소 스트레스 인지 정도: 28건 (5.4%)
  한번에 마시는 음주량: 26건 (5.0%)
  이상지질혈증 약복용 여부: 24건 (4.6%)
  고콜레스테롤혈증 유병여부: 21건 (4.0%)
  현재 일반담배 흡연 여부: 21건 (4.0%)
  고혈압 유병여부: 19건 (3.7%)
  유산소 신체활동 실천율: 19건 (3.7%)
  혈압조절제 복용 여부: 18건 (3.5%)
  당뇨병 유병여부: 18건 (3.5%)

▶ 대응 방안 제언
  null/empty 반환    → 프롬프트 개선 (값 필수 반환 지시 강화)
  형식 불일치(수치형) → Guided Decoding(--guided-json) 적용 권장
  형식 불일치(범주형) → 허용값 목록(Enum) 명시 또는 후처리 정규화

결과 저장: extraction_failure_classified.csv
